In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap

df = pd.read_csv('incident records.csv')

In [22]:
# Converting to 1/0

df['nkill'] = df['nkill'].astype(int)
df['nwound'] = df['nwound'].astype(int)

In [23]:
df

,iyear,imonth,iday,region_txt,country_txt,latitude,longitude,nwound,nkill
0,2000,1,1,North America,United States,44.405705,-85.714454,0,0
1,2000,1,2,South America,Colombia,4.667128,-74.106056,0,0
2,2000,1,2,South America,Colombia,7.198606,-75.341218,0,0
3,2000,1,3,North America,United States,38.232471,-122.644448,0,0
4,2000,1,3,North America,United States,39.103175,-84.511981,0,0
...,...,...,...,...,...,...,...,...,...
2992,2017,12,19,North America,Mexico,17.949243,-94.916882,0,1
2993,2017,12,20,South America,Colombia,7.745807,-76.556143,0,1
2994,2017,12,22,North America,United States,40.262772,-76.881107,0,0
2995,2017,12,22,North America,United States,40.261864,-76.880913,1,1


In [24]:
# Random Forest
features = ['iyear', 'region_txt', 'country_txt', 'nwound']
X = df[features]
y = df['nkill']

X_encoded = pd.get_dummies(X, drop_first=False)

X = pd.get_dummies( X, columns=["region_txt", "country_txt"], drop_first=True, dtype=int )

X_train, X_test, y_train, y_test, = train_test_split( 
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("Random forest test accuracy:", rf.score(X_test, y_test))

Random forest test accuracy: 0.695


In [31]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer(X_test)

shap_vals = shap_values[1]

country_cols = [col for col in X_test.columns if col.startswith("country_txt_")]

country_shap = shap_vals[:, X_test.columns.isin(country_cols)].sum(axis=1)

country_contribution = pd.DataFrame({
    "country_txt": df.loc[X_test.index, "country_txt"],
    "country_shap": country_shap
})

country_contribution.head()

IndexError: boolean index did not match indexed array along axis 1; size of axis is 2 but size of corresponding boolean axis is 16

In [25]:
# Build explainer
explainer = shap.TreeExplainer(rf)

shap_values = explainer(X_test)

print(shap_values.values.shape)

values = shap_values.values[0, :, 1]

for feature, value in zip(features, values):
    print(f"{feature}: {value:.2f}")

(600, 16, 2)
iyear: -0.05
region_txt: 0.24
country_txt: -0.01
nwound: 0.00


In [ ]:
explainer = shap.TreeExplainer(rf_model)
# Compute SHAP values for every row in the test set
shap_values = explainer(X_test)

print(shap_values.values.shape)

values = shap_values.values[0, :, 1]

for feature, value in zip(rf_features, values):
    print(f"{feature}: {value:.2f}")